# EV Charging & Adoption Dashboard (2010 - 2030)


## How to read this notebook

The notebook is organized as a **six-tab story**, each resolved a big question direction.

| Tab | Theme | Question it answers |
|---|---|---|
| **0. Overview** | Where are we today? | What does the global EV picture look like in one glance? |
| **1. Adoption Trends** | Who is winning? | Which countries grew fastest, and when did growth go exponential? |
| **2. Infrastructure & Demand** | Chicken or egg? | Do chargers lead EVs, or follow them? |
| **3. Market Composition** | What kind of EV? | BEV vs PHEV split, and is fleet turnover fast enough for 2030? |
| **4. Socioeconomic & Policy** | Is adoption equitable? | How does income shape EV uptake, and who are the outliers? |
| **5. Forecasting** | Where are we headed? | What does 2030 look like, and who hits the 30% threshold? |


## Setup

We use:

- **pandas / numpy** for data wrangling
- **plotly** for interactive charts (temporary before using Shiny library)
- **statsmodels** for ARIMA forecasting with prediction intervals (Prophet optional, compared with Projection-STEPS)

In [402]:
# !pip install pandas numpy plotly statsmodels --quiet

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

pio.templates.default = "plotly_white"

# Brand palette
PALETTE = {
    "primary":   "#0B7A75",   # teal — historical / hero series
    "accent":    "#F4A261",   # amber — projection / highlight
    "bev":       "#264653",   # deep slate — BEV
    "phev":      "#E76F51",   # coral — PHEV
    "fcev":      "#9C89B8",   # lavender — FCEV (rare)
    "fast":      "#2A9D8F",   # green-teal — fast charger
    "slow":      "#A8DADC",   # pale teal — slow charger
    "muted":     "#94A3B8",   # gray — reference / context
    "danger":    "#D00000",
    "warning":   "#FFB703",
    "success":   "#06A77D",
}

REGION_COLORS = {
    "World": PALETTE["muted"],
    "China": "#D62828",
    "USA": "#1D4E89",
    "EU27": "#003566",
    "Europe": "#005F73",
    "India": "#F77F00",
    "Norway": "#0077B6",
    "Germany": "#FFB703",
    "France": "#7209B7",
    "United Kingdom": "#9D0208",
    "Japan": "#BC4749",
    "Korea": "#FF006E",
    "Brazil": "#3A5A40",
    "Australia": "#FB8500",
    "Canada": "#9B2226",
}

# Page-level CSS for HTML-exported charts
PAGE_FONT = "Arial, Helvetica, sans-serif"

pio.templates["dashboard"] = go.layout.Template(
    layout=go.Layout(
        font=dict(
            family=PAGE_FONT,
            size=12,
            color="#222222"
        ),
        title=dict(
            font=dict(
                family=PAGE_FONT,
                size=20,
                color="#111111"
            )
        ),
        legend=dict(
            font=dict(
                family=PAGE_FONT,
                size=12
            )
        ),

        hoverlabel=dict(
            font=dict(
                family=PAGE_FONT
            )
        )
    )
)

pio.templates.default = "dashboard"

print("Environment ready. Plotly:", pio.templates.default)

Environment ready. Plotly: dashboard


## Load raw data

Quick shape & schema check before we transform anything.

In [403]:
CSV_PATH = "/content/EVDataExplorer2025.csv"

raw = pd.read_csv(CSV_PATH)
print(f"Rows: {len(raw):,}\nColumns: {raw.shape[1]}")
print("Columns:", list(raw.columns))
print("\n")
raw.head()


Rows: 16,436
Columns: 9
Columns: ['region_country', 'category', 'parameter', 'mode', 'powertrain', 'year', 'unit', 'value', 'Aggregate group']




,region_country,category,parameter,mode,powertrain,year,unit,value,Aggregate group
0,World,Projection-STEPS,EV stock,2 and 3 wheelers,BEV,2030,Vehicles,170000000.0,_World
1,World,Projection-STEPS,EV stock,Cars,BEV,2030,Vehicles,150000000.0,_World
2,China,Projection-STEPS,EV stock,2 and 3 wheelers,BEV,2030,Vehicles,91000000.0,Other
3,China,Projection-STEPS,EV stock,Cars,BEV,2030,Vehicles,82000000.0,Other
4,World,Projection-STEPS,EV stock,Cars,PHEV,2030,Vehicles,82000000.0,_World


In [404]:
print("Distinct values per categorical column:")
for c in raw.select_dtypes(include="object").columns:
    n = raw[c].nunique()
    sample = list(raw[c].dropna().unique()[:6])
    print(f"  {c:20s}  n={n:<5}  e.g. {sample}")

Distinct values per categorical column:
  region_country        n=63     e.g. ['World', 'China', 'Asia Pacific', 'India', 'Europe', 'Rest of the world']
  category              n=2      e.g. ['Projection-STEPS', 'Historical']
  parameter             n=9      e.g. ['EV stock', 'EV sales', 'EV charging points', 'Electricity demand', 'Oil displacement, million lge', 'Battery demand']
  mode                  n=6      e.g. ['2 and 3 wheelers', 'Cars', 'Vans', 'EV', 'Trucks', 'Buses']
  powertrain            n=6      e.g. ['BEV', 'PHEV', 'Publicly available slow', 'Publicly available fast', 'EV', 'FCEV']
  unit                  n=6      e.g. ['Vehicles', 'charging points', 'GWh', 'Oil displacement, million lge', 'percent', 'Million barrels per day']
  Aggregate group       n=4      e.g. ['_World', 'Other', 'Aggregate_sales_stock', 'EU27']


## Preprocessing

Following the spec literally:

| Step | Action |
|---|---|
| 1 | Rename column `Aggregate group` → `agg_group` |
| 2 | Map `agg_group` values: `_World`→`World`, `Aggregate_sales_stock`→`Continent`, `Other`→`Country`, (keep `EU27`) |
| 3 | Drop projection rows where `year < 2030` (keep only the 2030 end-point) |
| 4 | Rename `powertrain`: `Publicly available fast`→`Fast Charger`, `Publicly available slow`→`Slow Charger` |
| 5 | Rename `unit`: `Oil displacement, million lge`→`Oil displacement (Mlge)` |
| 6 | Rename `parameter`: `Oil displacement Mbd`→`Oil displacement (Mbd)`, `Oil displacement, million lge`→`Oil displacement (Mlge)` |

Add a binary column based on year: `is_projection`.

In [405]:
def preprocess(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # 1. rename column
    df = df.rename(columns={"Aggregate group": "agg_group"})

    # 2. remap agg_group values
    df["agg_group"] = df["agg_group"].replace({
        "_World": "World",
        "Aggregate_sales_stock": "Continent",
        "Other": "Country"
    })

    # 3. drop projection rows with year < 2030 (keep only 2030 horizon)
    drop_mask = (df["category"] == "Projection-STEPS") & (df["year"] < 2030)
    df = df.loc[~drop_mask].copy()

    # 4. rename powertrain values
    df["powertrain"] = df["powertrain"].replace({
        "Publicly available fast": "Fast Charger",
        "Publicly available slow": "Slow Charger",
    })

    # 5. rename unit
    df["unit"] = df["unit"].replace({
        "Oil displacement, million lge": "Oil displacement (Mlge)"
    })

    # 6. rename parameter
    df["parameter"] = df["parameter"].replace({
        "Oil displacement Mbd": "Oil displacement (Mbd)",
        "Oil displacement, million lge": "Oil displacement (Mlge)",
    })

    # convenience flag
    df["is_projection"] = df["category"] == "Projection-STEPS"

    return df.reset_index(drop=True)


df = preprocess(raw)
print(f"After preprocessing: {len(df):,} rows  ({len(raw)-len(df):,} dropped)")
print()
print("agg_group :", sorted(df["agg_group"].dropna().unique().tolist()))
print("powertrain:", sorted(df["powertrain"].dropna().unique().tolist()))
print("unit      :", sorted(df["unit"].dropna().unique().tolist()))
print("parameter :", sorted(df["parameter"].dropna().unique().tolist()))
print("years     :", df["year"].min(), "–", df["year"].max())

After preprocessing: 14,962 rows  (1,474 dropped)

agg_group : ['Continent', 'Country', 'EU27', 'World']
powertrain: ['BEV', 'EV', 'FCEV', 'Fast Charger', 'PHEV', 'Slow Charger']
unit      : ['GWh', 'Million barrels per day', 'Oil displacement (Mlge)', 'Vehicles', 'charging points', 'percent']
parameter : ['Battery demand', 'EV charging points', 'EV sales', 'EV sales share', 'EV stock', 'EV stock share', 'Electricity demand', 'Oil displacement (Mbd)', 'Oil displacement (Mlge)']
years     : 2010 – 2030


## Schema semantics

The same `powertrain` column means **three different things** depending on which `parameter` you're slicing.

### A. Partition of EV types
- `powertrain` belongs to {BEV, PHEV, FCEV}
- `mode` belongs to {Cars, Buses, Vans, Trucks, 2-and-3-wheelers}
- `unit` = Vehicles
- `parameter` belongs to {EV stock / EV sales / Battery demand / Oil displacement}

### B. Aggregated EV figures
- `powertrain` = EV
- `mode` belongs to {Cars, Buses, Vans, Trucks, 2-and-3-wheelers}
- `unit` = GWh or percent
- `parameter` belongs to {Electricity demand / EV stock share / EV sales share}

### C. Chargers
- `powertrain` belongs to {Fast Charger, Slow Charger}
- `mode` = EV
- `unit` = charging points
- `parameter` = EV charging point

**Rule of thumb:** before summing, always check both `parameter` AND `powertrain` to avoid mixing vehicles vs. charger vs. aggregated rollups.

In [406]:
# Check three groups
group_a = df[df["parameter"].isin(["EV stock","EV sales","Battery demand",
                                    "Oil displacement (Mbd)","Oil displacement (Mlge)"])]
group_b = df[df["parameter"].isin(["Electricity demand","EV stock share","EV sales share"])]
group_c = df[df["parameter"] == "EV charging points"]

print("Group A (vehicles) powertrains :", sorted(group_a["powertrain"].unique().tolist()))
print("Group B (aggregated) powertrains:", sorted(group_b["powertrain"].unique().tolist()))
print("Group C (chargers) powertrains:", sorted(group_c["powertrain"].unique().tolist()))


Group A (vehicles) powertrains : ['BEV', 'EV', 'FCEV', 'PHEV']
Group B (aggregated) powertrains: ['EV']
Group C (chargers) powertrains: ['Fast Charger', 'Slow Charger']


## Helper functions

In [407]:
def slice_data(df, parameter=None, mode=None, powertrain=None,
               region=None, agg_group=None, category=None, year=None):
    """Filter data based on columns and returns a copy."""
    out = df
    for col, val in [("parameter", parameter), ("mode", mode), ("powertrain", powertrain),
                     ("region_country", region), ("agg_group", agg_group),
                     ("category", category), ("year", year)]:
        if val is None:
            continue
        if isinstance(val, (list, tuple, set)):
            out = out[out[col].isin(val)]
        else:
            out = out[out[col] == val]
    return out.copy()


def total_ev_stock_by(df, group_cols, modes=("Cars",)):
    """Sum all types of EV for given modes, group by columns and returns DataFrame"""
    sub = slice_data(df, parameter="EV stock",
                     mode=list(modes),
                     powertrain=["BEV","PHEV","FCEV"])
    return sub.groupby(list(group_cols), as_index=False)["value"].sum()


def total_chargers_by(df, group_cols):
    """Total two types of chargers and group by columns"""
    sub = slice_data(df, parameter="EV charging points",
                     powertrain=["Fast Charger","Slow Charger"])
    return sub.groupby(list(group_cols), as_index=False)["value"].sum()


# Check
demo = total_ev_stock_by(df, ["region_country","year"]).query("region_country == 'World'")
print("World EV car stock (BEV+PHEV+FCEV) over time:")
print(demo.sort_values("year").to_string(index=False))

World EV car stock (BEV+PHEV+FCEV) over time:
region_country  year       value
         World  2010     20426.0
         World  2011     67526.0
         World  2012    190026.0
         World  2013    390027.0
         World  2014    710087.0
         World  2015   1250810.0
         World  2016   2003100.0
         World  2017   3106800.0
         World  2018   5111000.0
         World  2019   7219000.0
         World  2020  10226000.0
         World  2021  16342000.0
         World  2022  26057000.0
         World  2023  40066000.0
         World  2024  58070000.0
         World  2030 232220000.0


---

## TAB 0: General Information

> *"In 2010, electric cars were a curiosity. By 2024 they're a fifth of new sales. By 2030 the IEA expects them to be roughly 30%."*

In [408]:
LATEST_HIST = int(df.query("category == 'Historical'")["year"].max())
HORIZON     = 2030

def kpi(label, value, sub=""):
    return dict(label=label, value=value, sub=sub)

# KPI 1: total EV cars on the road, world, latest year
stock_world_latest = total_ev_stock_by(df, ["region_country","year"]) \
    .query("region_country == 'World' and year == @LATEST_HIST")["value"].sum()

# KPI 2: EV stock share, world, latest year
stock_share_latest = slice_data(df, parameter="EV stock share",
                                 region="World", mode="Cars",
                                 powertrain="EV", year=LATEST_HIST)["value"].sum()

# KPI 3: EV sales share, world, latest year
sales_share_latest = slice_data(df, parameter="EV sales share",
                                 region="World", mode="Cars",
                                 powertrain="EV", year=LATEST_HIST)["value"].sum()

# KPI 4: total public chargers, world, latest year
chargers_world_latest = total_chargers_by(df, ["region_country","year"]) \
    .query("region_country == 'World' and year == @LATEST_HIST")["value"].sum()

# KPI 5: projected world EV car stock in 2030
stock_world_2030 = total_ev_stock_by(df, ["region_country","year"]) \
    .query("region_country == 'World' and year == @HORIZON")["value"].sum()

kpis = [
    kpi(f"EV cars on the road, {LATEST_HIST}",   f"{stock_world_latest/1e6:.1f}M"),
    kpi(f"Share of car stock, {LATEST_HIST}",     f"{stock_share_latest:.1f}%"),
    kpi(f"Share of new car sales, {LATEST_HIST}", f"{sales_share_latest:.1f}%"),
    kpi(f"Public chargers, {LATEST_HIST}",        f"{chargers_world_latest/1e6:.2f}M"),
    kpi("Projected EV stock, 2030",         f"{stock_world_2030/1e6:.0f}M"),
]

# Render KPIs as a Plotly indicator strip
fig_kpi = make_subplots(rows=1, cols=5, specs=[[{"type":"indicator"}]*5])
for i, k in enumerate(kpis, start=1):
    fig_kpi.add_trace(go.Indicator(
        mode="number",
        value=float(k["value"].replace("M","").replace("%","")),
        number={"suffix": "M" if "M" in k["value"] else ("%" if "%" in k["value"] else ""),
                "font": {"size": 36, "color": PALETTE["primary"]}},
        title={"text": f"<span style='font-size:12px;color:#475569'>{k['label']}</span>"},
    ), row=1, col=i)

fig_kpi.update_layout(height=180, margin=dict(l=5, r=200, t=20, b=5),
                      paper_bgcolor="white",
                      title=dict(text=f"<b>Global EV scorecard</b>",
                      x=0, y=0.9, font=dict(size=16)))
fig_kpi.show()


In [409]:
nv_stock = df[(df["parameter"]=="EV stock") & (df["year"]==LATEST_HIST) & (df["agg_group"]=="Country")]
heat = nv_stock.groupby(["mode","powertrain"])["value"].median().reset_index()
heat_piv = heat.pivot(index="mode", columns="powertrain", values="value").fillna(0)
heat_piv = heat_piv.reindex(columns=["BEV","PHEV","FCEV"])

fig1c = go.Figure(go.Heatmap(
    z=np.log10(heat_piv.values),
    x=heat_piv.columns, y=heat_piv.index,
    text=[[f"{v:,.0f}" for v in row] for row in heat_piv.values],
    texttemplate="%{text}", textfont=dict(size=12),
    colorscale="tempo",
    hovertemplate="<b>%{y}</b> × %{x}<br>Median: %{text} vehicles<extra></extra>",
    colorbar=dict(title="log10(vehicles)", thickness=12)
))

fig1c.update_layout(height=400, margin=dict(t=60,b=60,l=50,r=500),
                    title=dict(x=0.02, y=0.95, xanchor="left", yanchor="top",
                               text=f"<b>EV stocks in average by mode and powertrain, 2024</b>"))
fig1c.show()

In [410]:
stock = df[(df["parameter"]=="EV stock") & (df["year"]==LATEST_HIST) & (df["agg_group"]=="Country")]
sales = df[(df["parameter"]=="EV sales") & (df["year"]==LATEST_HIST) & (df["agg_group"]=="Country")]
stock_agg = stock.groupby(["region_country","mode"], as_index=False)["value"].sum().rename(columns={"value":"stock"})
sales_agg = sales.groupby(["region_country","mode"], as_index=False)["value"].sum().rename(columns={"value":"sales"})
cluster = stock_agg.merge(sales_agg, on=["region_country","mode"], how="outer").fillna(0)
cluster = cluster[(cluster["stock"]>0) & (cluster["sales"]>0)].copy()
cluster["sales_to_stock"] = np.round(cluster["sales"] / cluster["stock"], 2)

fig1d = px.scatter(
    cluster, x="stock", y="sales", color="mode",
    hover_name="region_country", log_x=True, log_y=True,
    size="sales_to_stock", size_max=20,
    labels={"stock":"EV stock (log)","sales":"EV sales (log)","mode":"Mode"},
    title=f"<b>EV sales and actual on the road, 2024</b>"
)
fig1d.update_layout(height=600, margin=dict(t=60,b=60,l=0,r=500),
                    title=dict(x=0.02, y=0.95, xanchor="left", yanchor="top"),
                    legend=dict(title=None, orientation="h", x=0.4, y=1, xanchor="left", yanchor="top"))
fig1d.show()

In [411]:
# ---- Global trajectory: stock + sales share ----

# stock series, historical + 2030 projection
stock_series = total_ev_stock_by(df, ["region_country","year","category"]) \
    .query("region_country == 'World'")
stock_hist = stock_series[stock_series["category"]=="Historical"].sort_values("year")
stock_proj = stock_series[stock_series["category"]=="Projection-STEPS"].sort_values("year")

# sales-share series
share = slice_data(df, parameter="EV sales share",
                    region="World", mode="Cars", powertrain="EV")
share_hist = share[share["category"]=="Historical"].sort_values("year")
share_proj = share[share["category"]=="Projection-STEPS"].sort_values("year")

fig0 = make_subplots(specs=[[{"secondary_y": True}]])

# stock area (historical)
fig0.add_trace(go.Scatter(
    x=stock_hist["year"], y=stock_hist["value"]/1e6,
    fill="tozeroy", mode="lines", name="EV car stock (2010-2024)",
    line=dict(color=PALETTE["primary"], width=2.5),
    fillcolor="rgba(11,122,117,0.18)",
    hovertemplate="%{y}M EV cars<extra></extra>",
), secondary_y=False)

# stock projection bridge (2030)
bridge_x = [stock_hist["year"].iloc[-1]] + stock_proj["year"].tolist()
bridge_y = [stock_hist["value"].iloc[-1]] + stock_proj["value"].tolist()
fig0.add_trace(go.Scatter(
    x=bridge_x, y=[v/1e6 for v in bridge_y],
    mode="lines+markers", name="EV car stock (2030)",
    line=dict(color=PALETTE["primary"], width=2.5, dash="dash"),
    hovertemplate="%{y}M EV cars (projected)<extra></extra>",
), secondary_y=False)

# sales share line on secondary axis
fig0.add_trace(go.Scatter(
    x=share_hist["year"], y=share_hist["value"],
    fill="tozeroy", mode="lines", name="EV sales share (2010-2024)",
    line=dict(color=PALETTE["bev"], width=2.5),
    fillcolor="rgba(38,70,83,0.18)",
    hovertemplate="%{y}% of new cars<extra></extra>",
), secondary_y=True)

if len(share_proj):
    fig0.add_trace(go.Scatter(
        x=[share_hist["year"].iloc[-1]] + share_proj["year"].tolist(),
        y=[share_hist["value"].iloc[-1]] + share_proj["value"].tolist(),
        mode="lines+markers", name="EV sales share (2030)",
        line=dict(color=PALETTE["bev"], width=2, dash="dot"),
        hovertemplate="%{y}% of new cars (projected)<extra></extra>"
    ), secondary_y=True)

fig0.update_layout(height=500, margin=dict(t=60,b=60,l=0,r=200), xaxis_title="Year",
                    title=dict(x=0.02, y=0.95, xanchor="left", yanchor="top",
                               text=f"<b>Current and projected trends of EV Car Fleet & Sales Share</b>"),
                    legend=dict(title=None, orientation="h", x=0.4, y=1, xanchor="left", yanchor="top"))

fig0.update_yaxes(title_text="EV cars on the road (millions)", secondary_y=False)
fig0.update_yaxes(title_text="Share of new car sales (%)", secondary_y=True)
fig0.show()

In [412]:
nv_world = df[(df["agg_group"]=="World") & (df["parameter"]=="EV stock") & (df["mode"]=="Cars")]
nv_world_pt = nv_world.groupby(["year","powertrain"], as_index=False)["value"].sum()

fig1e = px.area(
    nv_world_pt.sort_values("year"), x="year", y="value", color="powertrain",
    labels={"value":"EV stock (vehicles)","year":"Year","powertrain":"Powertrain"},
    title="<b>World EV car stocks by powertrain over time</b>",
)

fig1e.update_layout(height=400, margin=dict(t=60,b=60,l=0,r=500),
                    title=dict(x=0.02, y=0.95, xanchor="left", yanchor="top"),
                    legend=dict(title=None, orientation="h", x=0.5, y=1.05, xanchor="left", yanchor="top"))
fig1e.show()

In [413]:
ch_world = df[(df["agg_group"]=="World") & (df["unit"]=="charging points")].groupby(["year","powertrain"], as_index=False)["value"].sum()

fig3a = px.bar(
    ch_world.sort_values("year"), x="year", y="value", color="powertrain", barmode="stack",
    labels={"value":"Charging points","year":"Year","powertrain":"Charger type"},
    title="<b>World charging points over time</b>",
)
fig3a.update_layout(height=400, margin=dict(t=60,b=60,l=0,r=500),
                    title=dict(x=0.02, y=0.95, xanchor="left", yanchor="top"),
                    legend=dict(title=None, orientation="h", x=0.5, y=1.05, xanchor="left", yanchor="top"))
fig3a.show()

**Reading the chart.** The global stock of EV cars on the road remained plateau through 2015, took off after 2018, and finally reached 58.1M by 2024, with the STEPS scenario projecting 232M by 2030. Meanwhile, the share of eletric car sales leads stock by years: by 2024 roughly one in five new cars sold is electric, even though only one in twenty cars on the road is.

---

## TAB 1: Adoption Trends

### Chart 1.1: Animated choropleth: EV stock share of cars, by country, 2010 → 2024

The map answers "who is winning?" at a glance. Drag the slider to watch Norway turn dark first, then the Netherlands, then the rest of Europe, then China surging in volume.

In [414]:
# ---- Animated choropleth: EV stock share by country from 2010 to 2024 ----

# We use EV stock share (parameter='EV stock share', mode='Cars', powertrain='EV')
# Restricted to countries (agg_group == 'Country') so we don't double-paint continents
share_map = slice_data(df,
                       parameter="EV stock share",
                       mode="Cars", powertrain="EV",
                       agg_group="Country",
                       category="Historical")
share_map = share_map[(share_map["year"] >= 2010) & (share_map["year"] <= LATEST_HIST)].copy()

# Plotly choropleth needs ISO-3 country codes. The dataset uses common names; map them.
ISO3 = {
    "Australia":"AUS","Austria":"AUT","Belgium":"BEL","Brazil":"BRA","Bulgaria":"BGR",
    "Canada":"CAN","Chile":"CHL","China":"CHN","Colombia":"COL","Costa Rica":"CRI",
    "Croatia":"HRV","Cyprus":"CYP","Czech Republic":"CZE","Denmark":"DNK","Estonia":"EST",
    "Finland":"FIN","France":"FRA","Germany":"DEU","Greece":"GRC","Hungary":"HUN",
    "Iceland":"ISL","India":"IND","Indonesia":"IDN","Ireland":"IRL","Israel":"ISR",
    "Italy":"ITA","Japan":"JPN","Korea":"KOR","Latvia":"LVA","Lithuania":"LTU",
    "Luxembourg":"LUX","Malta":"MLT","Mexico":"MEX","Netherlands":"NLD","New Zealand":"NZL",
    "Norway":"NOR","Poland":"POL","Portugal":"PRT","Romania":"ROU","Slovakia":"SVK",
    "Slovenia":"SVN","South Africa":"ZAF","Spain":"ESP","Sweden":"SWE","Switzerland":"CHE",
    "Thailand":"THA","Turkey":"TUR","Turkiye":"TUR","Ukraine":"UKR",
    "United Kingdom":"GBR","United States":"USA",
}
share_map["iso3"] = share_map["region_country"].map(ISO3)
share_map = share_map.dropna(subset=["iso3"])

# fill missing year-country combos with 0 so the slider doesn't blank countries
years = sorted(share_map["year"].unique())
all_iso = share_map["iso3"].unique()
grid = pd.MultiIndex.from_product([years, all_iso], names=["year","iso3"]).to_frame(index=False)
grid = grid.merge(share_map[["iso3","region_country"]].drop_duplicates(), on="iso3")
share_map_full = grid.merge(share_map[["iso3","year","value"]], on=["iso3","year"], how="left")
share_map_full["value"] = share_map_full["value"].fillna(0)

fig_map = px.choropleth(
    share_map_full.sort_values("year"),
    locations="iso3", color="value",
    hover_name="region_country",
    animation_frame="year",
    color_continuous_scale=[
        [0,    "#F1FAEE"],
        [0.05, "#A8DADC"],
        [0.20, "#457B9D"],
        [0.50, "#1D3557"],
        [1.00, "#0B0B3B"],
    ],
    range_color=[0, share_map_full["value"].quantile(0.98)],
    labels={"value":"EV stock share (%)"},
    title=f"<b>EV stock share of car fleet by country from 2010 to {LATEST_HIST}</b>",
)
fig_map.update_geos(showcountries=True, countrycolor="#CBD5E1",
                    showcoastlines=False, projection_type="natural earth")
fig3a.update_layout(height=400, margin=dict(t=60,b=60,l=0,r=500),
                    title=dict(x=0.02, y=0.95, xanchor="left", yanchor="top"),
                    legend=dict(title=None, orientation="h", x=0.5, y=1.05, xanchor="left", yanchor="top"))
fig_map.show()


### Chart 1.2: Race bar: top-15 country EV stock

Where the choropleth shows *intensity* (share), the race bar shows *volume*. Watch China's bar swallow the chart from 2018 onward.

In [415]:
stock_country = total_ev_stock_by(df, ["region_country","year","category"])
stock_country = stock_country[(stock_country["category"]=="Historical") &
                              (stock_country["year"] >= 2010)]
# keep only true countries (drop World/continents/EU27)
stock_country = stock_country.merge(
    df[["region_country","agg_group"]].drop_duplicates(),
    on="region_country", how="left"
)
stock_country = stock_country[stock_country["agg_group"] == "Country"].copy()

# for each year keep top 15
frames = []
for y in sorted(stock_country["year"].unique()):
    sub = stock_country[stock_country["year"] == y].nlargest(15, "value").sort_values("value")
    sub = sub.assign(year=y)
    frames.append(sub)
race_df = pd.concat(frames, ignore_index=True)
race_df["value_M"] = race_df["value"] / 1e6

def race_color(c):
    return REGION_COLORS.get(c, PALETTE["primary"])

fig_race = go.Figure()
years_race = sorted(race_df["year"].unique())

# initial frame = first year
init = race_df[race_df["year"] == years_race[0]]
fig_race.add_trace(go.Bar(
    x=init["value_M"], y=init["region_country"], orientation="h",
    marker=dict(color=[race_color(c) for c in init["region_country"]]),
    text=[f"{v:.2f} M" for v in init["value_M"]],
    textposition="outside",
    hovertemplate="<b>%{y}</b>: %{x:.2f} M EVs<extra></extra>",
))

# build frames
fig_frames = []
for y in years_race:
    sub = race_df[race_df["year"] == y]
    fig_frames.append(go.Frame(
        name=str(y),
        data=[go.Bar(
            x=sub["value_M"], y=sub["region_country"], orientation="h",
            marker=dict(color=[race_color(c) for c in sub["region_country"]]),
            text=[f"{v:.2f} M" for v in sub["value_M"]],
            textposition="outside",
        )],
        layout=go.Layout(title_text=f"<b>Top-15 EV car stock by country — {y}</b>")
    ))
fig_race.frames = fig_frames

fig_race.update_layout(
    title=f"<b>Top-15 EV car stock by country — {years_race[0]}</b>",
    height=560, xaxis_title="EV cars on the road (millions)", yaxis_title="",
    margin=dict(l=140,r=80,t=70,b=80),
    updatemenus=[{
        "type": "buttons", "showactive": False, "y": -0.2, "x": 0,
        "direction": "right", "direction": "right",
        "buttons": [
            {"label":"▶",  "method":"animate",
             "args":[None, {"frame":{"duration":700,"redraw":True}, "fromcurrent":True}]},
            {"label":"❚❚","method":"animate",
             "args":[[None], {"frame":{"duration":0,"redraw":False}, "mode":"immediate"}]},
        ],
    }],
    sliders=[{
        "active":0, "y":-0.10, "x":0.05, "len":0.92,
        "currentvalue":{"prefix":"Year: ","font":{"size":14,"color":PALETTE["primary"]}},
        "steps":[{"label":str(y), "method":"animate",
                  "args":[[str(y)], {"frame":{"duration":500,"redraw":True},
                                     "mode":"immediate"}]} for y in years_race],
    }],
)
fig_race.show()


### Chart 1.3: China vs United States EV sales

In [416]:
sales = slice_data(df, parameter="EV sales",
                   mode="Cars", powertrain=["BEV","PHEV","FCEV"],
                   region=["China", "USA"],
                   category="Historical")
sales = sales.groupby(["region_country","year"], as_index=False)["value"].sum()

# policy event markers
policy_events = [
  {"year": 2012, "country": "USA",   "label": "Tesla Model S released"},
  {"year": 2017, "country": "China", "label": "NEV credit mandate announced"},
  {"year": 2020, "country": "China", "label": "Mass charging infrastructure rollout"},
  {"year": 2021, "country": "USA",   "label": "Biden EV target announced"},
  {"year": 2023, "country": "China", "label": "BYD surpasses Tesla sales"},
  {"year": 2023, "country": "USA",   "label": "Tesla charging standard adopted"},
]

fig_infl = go.Figure()
for country in ["China", "USA"]:
    sub = sales[sales["region_country"] == country].sort_values("year")
    fig_infl.add_trace(go.Scatter(
        x=sub["year"], y=sub["value"],
        mode="lines+markers", name=country,
        line=dict(color=REGION_COLORS[country], width=3),
        marker=dict(size=8),
        hovertemplate=f"<b>{country}</b><br>%{{x}}: %{{y:,.0f}} EVs sold<extra></extra>",
    ))

# overlay policy events
for ev in policy_events:
    sub = sales[(sales["region_country"]==ev["country"]) & (sales["year"]==ev["year"])]
    if len(sub) == 0: continue
    y_val = sub["value"].iloc[0]
    fig_infl.add_annotation(
        x=ev["year"], y=y_val,
        text=f"<b>{ev['country'][:2]}</b> · {ev['label']}",
        showarrow=True, arrowhead=2, arrowcolor=PALETTE["muted"],
        ax=0, ay=-50, bgcolor="rgba(255,255,255,0.92)",
        bordercolor=PALETTE["muted"], borderpad=4,
        font=dict(size=10, color="#1f2937"),
    )

fig_infl.update_yaxes(type="linear", title_text="EV sales (vehicles, log scale)")
fig_infl.update_xaxes(title_text="Year")
fig_infl.update_layout(
    title=dict(x=0.02, y=0.98, xanchor="left", yanchor="top",
               text="<b>When did exponential growth start?</b>"),
    height=500, hovermode="x unified",
    legend=dict(orientation="h", y=1, x=0.4, xanchor="left", yanchor="top"),
    margin=dict(t=60,b=60,l=50,r=300),
)

# fig3a.update_layout(height=400, margin=dict(t=60,b=60,l=0,r=500),
#                     title=dict(x=0.02, y=0.95, xanchor="left", yanchor="top"),
#                     legend=dict(title=None, orientation="h", x=0.5, y=1.05, xanchor="left", yanchor="top"))
fig_infl.show()


---

## TAB 2: Infrastructure & Demand

> *"Chickens or eggs? Do chargers come first, or EVs?"*

### Chart 2.1: total EVs vs total public chargers in the world

If chargers lead EVs, the charger curve should be steeper or earlier. If they lag, EVs lead.

In [417]:
# ---- Dual area chart: stock vs chargers, world ----

# Two area subplots stacked vertically so reader can compare *shapes* not magnitudes
ev_world = total_ev_stock_by(df, ["region_country","year","category"]) \
    .query("region_country == 'World' and category == 'Historical'") \
    .sort_values("year")
ch_world = total_chargers_by(df, ["region_country","year","category"]) \
    .query("region_country == 'World' and category == 'Historical'") \
    .sort_values("year")

# also break chargers into fast vs slow
ch_split = slice_data(df, parameter="EV charging points",
                       region="World", category="Historical",
                       powertrain=["Fast Charger","Slow Charger"])
ch_split = ch_split.groupby(["year","powertrain"], as_index=False)["value"].sum()

fig_inf = make_subplots(rows=2, cols=1, shared_xaxes=True,
                        subplot_titles=("EV cars on the road (BEV+PHEV+FCEV, all modes summed = Cars)",
                                        "Public chargers (Fast vs Slow)"),
                        vertical_spacing=0.12)

fig_inf.add_trace(go.Scatter(
    x=ev_world["year"], y=ev_world["value"]/1e6,
    fill="tozeroy", mode="lines", name="EV stock (M)",
    line=dict(color=PALETTE["primary"], width=2.5),
    fillcolor="rgba(11,122,117,0.22)",
    hovertemplate="%{x}: %{y:.2f} M EVs<extra></extra>",
), row=1, col=1)

for pt, color in [("Slow Charger", PALETTE["slow"]), ("Fast Charger", PALETTE["fast"])]:
    sub = ch_split[ch_split["powertrain"]==pt].sort_values("year")
    fig_inf.add_trace(go.Scatter(
        x=sub["year"], y=sub["value"]/1e6,
        mode="lines", name=pt,
        stackgroup="chargers",
        line=dict(color=color, width=1.5),
        fillcolor=color,
        hovertemplate=f"<b>{pt}</b> %{{x}}: %{{y:.2f}} M points<extra></extra>",
    ), row=2, col=1)

fig_inf.update_xaxes(title_text="Year", row=2, col=1)
fig_inf.update_yaxes(title_text="EVs (millions)", row=1, col=1)
fig_inf.update_yaxes(title_text="Charging points (millions)", row=2, col=1)
fig_inf.update_layout(
    title="<b>Chargers vs EVs: do they grow together?</b> · world, historical",
    height=620, hovermode="x unified",
    legend=dict(orientation="h", y=-0.10, x=0.5, xanchor="center"),
    margin=dict(l=60,r=60,t=80,b=70),
)
fig_inf.show()


### Chart 2.2: Charger-to-EV ratio: which countries are most under-built?

We compute **EVs per public charging point** (lower = better). Color = continent. Size = total EV stock.

In [418]:
latest = LATEST_HIST

ev_country = total_ev_stock_by(df, ["region_country","year","category"]) \
    .query("category == 'Historical' and year == @latest")

ch_country = total_chargers_by(df, ["region_country","year","category"]) \
    .query("category == 'Historical' and year == @latest")

ratio = ev_country.merge(
    ch_country,
    on=["region_country","year","category"],
    suffixes=["_ev","_ch"]
)

# filter to countries only
ratio = ratio.merge(
    df[["region_country","agg_group"]].drop_duplicates(),
    on="region_country"
)

ratio = ratio[
    ((ratio["agg_group"] == "Country") | (ratio["agg_group"] == "World")) &
    (ratio["value_ev"] >= 5_000) &
    (ratio["value_ch"] > 0)
].copy()

ratio["evs_per_charger"] = ratio["value_ev"] / ratio["value_ch"]
ratio["ev_stock_M"]      = ratio["value_ev"] / 1e6

# ---------------------------------------------------------
# continent lookup
# ---------------------------------------------------------
CONTINENT = {
    "China":"Asia","India":"Asia","Indonesia":"Asia","Japan":"Asia","Korea":"Asia",
    "Thailand":"Asia","Israel":"Asia","Turkey":"Asia","Turkiye":"Asia",

    "Germany":"Europe","France":"Europe","United Kingdom":"Europe","Norway":"Europe",
    "Sweden":"Europe","Netherlands":"Europe","Italy":"Europe","Spain":"Europe",
    "Belgium":"Europe","Switzerland":"Europe","Denmark":"Europe","Finland":"Europe",
    "Poland":"Europe","Austria":"Europe","Portugal":"Europe","Ireland":"Europe",
    "Greece":"Europe","Czech Republic":"Europe","Iceland":"Europe","Hungary":"Europe",
    "Slovakia":"Europe","Slovenia":"Europe","Romania":"Europe","Bulgaria":"Europe",
    "Croatia":"Europe","Estonia":"Europe","Latvia":"Europe","Lithuania":"Europe",
    "Luxembourg":"Europe","Malta":"Europe","Cyprus":"Europe","Ukraine":"Europe",

    "United States":"America","Canada":"America","Mexico":"America",
    "Brazil":"America","Chile":"America","Colombia":"America","Costa Rica":"America",

    "South Africa":"Africa",

    "Australia":"Oceania","New Zealand":"Oceania",
}

ratio["continent"] = ratio["region_country"].map(CONTINENT)
ratio["continent"] = ratio["continent"].fillna("Other")

CONTINENT_COLORS = {
    "Asia":"#D62828",
    "Europe":"#1D4E89",
    "America":"#9B2226",
    "Africa":"#FB8500",
    "Oceania":"#7209B7",
    "Other":"#94A3B8",
}

# ---------------------------------------------------------
# marker size scaling
# ---------------------------------------------------------
ratio["bubble_size"] = np.sqrt(ratio["value_ev"]) / 120

# ---------------------------------------------------------
# labels inside bubbles for large circles
# ---------------------------------------------------------
ratio["text_inside"] = np.where(
    ratio["bubble_size"] >= 12,
    ratio["region_country"],
    ""
)

# ---------------------------------------------------------
# scatter
# ---------------------------------------------------------
fig_ratio = px.scatter(
    ratio.sort_values("evs_per_charger"),

    x="ev_stock_M",
    y="evs_per_charger",

    size="bubble_size",
    size_max=60,

    color="continent",
    color_discrete_map=CONTINENT_COLORS,

    hover_name="region_country",

    # IMPORTANT
    text="text_inside",

    labels={
        "ev_stock_M":"EV stock (M, log scale)",
        "evs_per_charger":"EVs per public charger",
        "continent":"Continent"
    },

    title=f"<b>Charger-to-EV stress test, {latest}</b>",
)

# ---------------------------------------------------------
# bubble styling
# ---------------------------------------------------------
fig_ratio.update_traces(
    marker=dict(
        line=dict(width=1.2, color="white"),
        opacity=0.82
    ),

    # center text inside bubble
    textposition="middle center",

    textfont=dict(
        size=10,
        color="white"
    )
)

fig_ratio.show()

In [419]:
worst = ratio.nlargest(5, "evs_per_charger")

print("\nTop-5 worst charger-to-EV ratios:\n")

print(
    worst[
        ["region_country","ev_stock_M","value_ch","evs_per_charger"]
    ]
    .rename(columns={"value_ch":"chargers"})
    .to_string(index=False)
)


Top-5 worst charger-to-EV ratios:

   region_country  ev_stock_M  chargers  evs_per_charger
      New Zealand    0.113038    1440.0        78.498611
Rest of the world    0.750041   12400.0        60.487177
        Australia    0.302089    6700.0        45.087910
           Mexico    0.068000    1850.0        36.756757
              USA    6.319000  193000.0        32.740933


**Tab 2 takeaways.**

- Globally, **chargers grew slightly faster than vehicles in 2018–22**, then the EV curve overtook charger growth. Chargers now appear to *lag* EV uptake at the global level.

---

## TAB 3: Market Composition

> *"A BEV and a PHEV both decarbonize, but they're different products with different infrastructure needs."*

### Chart 3.1: Stacked bar: BEV vs PHEV share by country, latest year

We rank the top 20 countries by total EV stock, then show the BEV/PHEV mix as a 100% stacked bar.

In [420]:
# ---- Stacked bar: BEV vs PHEV mix per country, latest year ----

mix = slice_data(df, parameter="EV stock", mode="Cars",
                  powertrain=["BEV","PHEV"],
                  category="Historical", year=LATEST_HIST,
                  agg_group="Country")
mix_p = mix.pivot_table(index="region_country", columns="powertrain",
                        values="value", aggfunc="sum", fill_value=0)
mix_p["total"] = mix_p.sum(axis=1)
mix_p = mix_p.nlargest(20, "total").sort_values("total")
mix_p["BEV_pct"]  = mix_p["BEV"]  / mix_p["total"] * 100
mix_p["PHEV_pct"] = mix_p["PHEV"] / mix_p["total"] * 100

fig_mix = go.Figure()
fig_mix.add_trace(go.Bar(
    y=mix_p.index, x=mix_p["BEV_pct"], orientation="h", name="BEV",
    marker_color=PALETTE["bev"],
    text=[f"{v:.0f}%" for v in mix_p["BEV_pct"]], textposition="inside",
    hovertemplate="<b>%{y}</b><br>BEV: %{x:.1f}%<extra></extra>",
))
fig_mix.add_trace(go.Bar(
    y=mix_p.index, x=mix_p["PHEV_pct"], orientation="h", name="PHEV",
    marker_color=PALETTE["phev"],
    text=[f"{v:.0f}%" for v in mix_p["PHEV_pct"]], textposition="inside",
    hovertemplate="<b>%{y}</b><br>PHEV: %{x:.1f}%<extra></extra>",
))
fig_mix.update_layout(barmode="stack", height=600,
                      title=dict(x=0.02, y=0.95, xanchor="left", yanchor="top",
                                 text=f"<b>Top 20 EV markets proportion, {LATEST_HIST}</b>"),
                      xaxis_title="Share of EV car stock (%)",
                      margin=dict(t=60,b=60,l=0,r=500),
                      legend=dict(title=None, orientation="h", x=0.5, y=1.07, xanchor="left", yanchor="top"))

# fig1d.update_layout(height=600, margin=dict(t=60,b=60,l=0,r=500),
#                     title=dict(x=0.02, y=0.95, xanchor="left", yanchor="top"),
#                     legend=dict(title=None, orientation="h", x=0.4, y=1, xanchor="left", yanchor="top"))
fig_mix.show()


### Chart 3.2: Fleet turnover: sales share vs stock share

If sales share rises *much faster* than stock share, the fleet is turning over fast enough to hit 2030 targets. If they're close, the legacy ICE fleet is dragging.

In [421]:
# ---- Fleet turnover: sales share vs stock share, world ----

sales_share = slice_data(df, parameter="EV sales share",
                          region="World", mode="Cars", powertrain="EV") \
    .sort_values("year")
stock_share = slice_data(df, parameter="EV stock share",
                          region="World", mode="Cars", powertrain="EV") \
    .sort_values("year")

fig_turn = go.Figure()
for series, name, color in [(sales_share, "Sales share (new cars)", PALETTE["primary"]),
                             (stock_share, "Stock share (cars on road)", PALETTE["accent"])]:
    hist = series[series["category"]=="Historical"]
    proj = series[series["category"]=="Projection-STEPS"]
    fig_turn.add_trace(go.Scatter(
        x=hist["year"], y=hist["value"],
        mode="lines+markers", name=name,
        line=dict(color=color, width=3),
        marker=dict(size=8),
    ))
    if len(proj):
        bridge_x = [hist["year"].iloc[-1]] + proj["year"].tolist()
        bridge_y = [hist["value"].iloc[-1]] + proj["value"].tolist()
        fig_turn.add_trace(go.Scatter(
            x=bridge_x, y=bridge_y,
            mode="lines+markers", name=f"{name} (STEPS 2030)",
            line=dict(color=color, width=2, dash="dash"),
            marker=dict(size=10, symbol="diamond"),
            showlegend=True,
        ))

fig_turn.update_layout(
    title=dict(x=0.02, y=0.95, xanchor="left", yanchor="top",
               text="<b>The turnover gap races between sales share and stock share</b>"),
    height=500, xaxis_title="Year", yaxis_title="% of cars",
    legend=dict(orientation="h", y=-0.18, x=0.5, xanchor="center"),
    hovermode="x unified", margin=dict(t=60,b=60,l=0,r=500),
)
fig_turn.show()



**Tab 3 takeaways.**

- **Korea are nearly pure-BEV** (Norway, Iceland >85% BEV).
- **China is BEV-dominant** despite the size of its PHEV market, because BEV volume scaled faster.

---

## TAB 4: Socioeconomic & Policy

> *"Is EV adoption equitable across income levels, or only a rich-country story?"*

We classify countries into **High / Upper-middle / Lower-middle income** groups (World Bank-style heuristics), then plot **EV penetration (sales share)** vs **charging infrastructure density (chargers per million people)**.

In [422]:
# ---- Scatter: EV penetration vs charging-infra density, by income group ----

# Coarse income classification (manual; would normally pull from WB API)
INCOME = {
    # High income
    "Norway":"High","Sweden":"High","Iceland":"High","Denmark":"High","Finland":"High",
    "Netherlands":"High","Germany":"High","France":"High","United Kingdom":"High",
    "Switzerland":"High","Austria":"High","Belgium":"High","Ireland":"High",
    "Italy":"High","Spain":"High","Portugal":"High","Greece":"High","Luxembourg":"High",
    "United States":"High","Canada":"High","Australia":"High","New Zealand":"High",
    "Japan":"High","Korea":"High","Israel":"High","Estonia":"High","Slovenia":"High",
    "Czech Republic":"High","Slovakia":"High","Croatia":"High","Cyprus":"High",
    "Malta":"High","Latvia":"High","Lithuania":"High","Poland":"High","Hungary":"High",
    # Upper-middle income
    "China":"Upper-middle","Brazil":"Upper-middle","Mexico":"Upper-middle",
    "South Africa":"Upper-middle","Turkey":"Upper-middle","Turkiye":"Upper-middle",
    "Romania":"Upper-middle","Bulgaria":"Upper-middle","Costa Rica":"Upper-middle",
    "Colombia":"Upper-middle","Chile":"Upper-middle","Thailand":"Upper-middle",
    # Lower-middle income
    "India":"Lower-middle","Indonesia":"Lower-middle","Ukraine":"Lower-middle",
}

# Country population (millions, ~2023, approximate; for density normalization)
POP_M = {
    "Norway":5.5,"Sweden":10.5,"Iceland":0.4,"Denmark":5.9,"Finland":5.6,"Netherlands":17.9,
    "Germany":84.5,"France":68.0,"United Kingdom":68.0,"Switzerland":8.8,"Austria":9.0,
    "Belgium":11.7,"Ireland":5.1,"Italy":59.0,"Spain":48.0,"Portugal":10.3,"Greece":10.4,
    "Luxembourg":0.7,"United States":335.0,"Canada":40.0,"Australia":26.6,"New Zealand":5.2,
    "Japan":125.0,"Korea":51.7,"Israel":9.7,"Estonia":1.3,"Slovenia":2.1,"Czech Republic":10.5,
    "Slovakia":5.4,"Croatia":3.9,"Cyprus":1.3,"Malta":0.5,"Latvia":1.9,"Lithuania":2.8,
    "Poland":36.7,"Hungary":9.6,"China":1411.0,"Brazil":216.0,"Mexico":129.0,
    "South Africa":60.4,"Turkey":85.4,"Turkiye":85.4,"Romania":19.0,"Bulgaria":6.4,
    "Costa Rica":5.2,"Colombia":52.0,"Chile":19.6,"Thailand":71.7,"India":1428.0,
    "Indonesia":278.0,"Ukraine":36.7,
}

pen = slice_data(df, parameter="EV sales share",
                  mode="Cars", powertrain="EV",
                  category="Historical", year=LATEST_HIST,
                  agg_group="Country")
ch_density = total_chargers_by(df, ["region_country","year","category"]) \
    .query("category == 'Historical' and year == @LATEST_HIST")

socio = pen[["region_country","value"]].rename(columns={"value":"sales_share"})
socio = socio.merge(ch_density[["region_country","value"]].rename(columns={"value":"chargers"}),
                    on="region_country", how="left")
socio["income"]      = socio["region_country"].map(INCOME)
socio["pop_M"]       = socio["region_country"].map(POP_M)
socio["ch_per_M"]    = socio["chargers"] / socio["pop_M"]
socio = socio.dropna(subset=["income","pop_M","chargers"])
socio = socio[socio["chargers"] > 0].copy()

INCOME_COLORS = {"High":"#0B7A75","Upper-middle":"#F4A261","Lower-middle":"#E76F51"}

fig_soc = px.scatter(
    socio, x="ch_per_M", y="sales_share",
    color="income", color_discrete_map=INCOME_COLORS,
    size="pop_M", size_max=55,
    hover_name="region_country",
    labels={"ch_per_M":"Public chargers per million people",
            "sales_share":"EV sales share of new cars (%)",
            "income":"Income group"},
    title=f"<b>Adoption vs infrastructure density, {LATEST_HIST}</b>"
)
# annotate outliers (high adoption AT each income group)
for grp in ["High","Upper-middle","Lower-middle"]:
    grp_df = socio[socio["income"]==grp]
    if len(grp_df) == 0: continue
    top = grp_df.nlargest(2, "sales_share")
    for _, r in top.iterrows():
        fig_soc.add_annotation(
            x=r["ch_per_M"], y=r["sales_share"],
            text=r["region_country"], showarrow=True, arrowhead=2,
            ax=20, ay=-15, font=dict(size=10))

fig_soc.update_layout(height=600, margin=dict(t=60,b=60,l=0,r=500),
                      title=dict(x=0.1, y=0.95, xanchor="left", yanchor="top"),
                       legend=dict(orientation="h", y=-0.18, x=0.5, xanchor="center"))
fig_soc.show()



In [423]:
# summary stats by income
print("\nMedian EV sales share by income group:")
print(socio.groupby("income")["sales_share"].median().round(1).sort_values(ascending=False))


Median EV sales share by income group:
income
High            24.0
Upper-middle     7.4
Lower-middle     4.7
Name: sales_share, dtype: float64


---

## TAB 5: Forecasting

> *"The IEA STEPS gives us a 2030 endpoint. Can we triangulate that with our own statistical model?"*

We fit an **ARIMA(1,2,1)** to the historical world EV stock series, generate a forecast with 80% & 95% prediction intervals, and overlay the IEA STEPS scenario for comparison.

ARIMA is chosen over Prophet here because: (a) the series is short (~15 points) so simple wins, (b) we have no clear seasonality (annual data), and (c) statsmodels is a lighter dependency.

In [424]:
# ---- ARIMA forecast: world EV stock 2025-2030 ----

from statsmodels.tsa.arima.model import ARIMA

hist_stock = total_ev_stock_by(df, ["region_country","year","category"]) \
    .query("region_country == 'World' and category == 'Historical'") \
    .sort_values("year").set_index("year")["value"]

# fit ARIMA(1,2,1) — second-difference captures the accelerating curve
model = ARIMA(hist_stock, order=(1, 2, 1))
res = model.fit()
print(res.summary().tables[1])

# forecast 6 steps ahead (2025..2030)
fc = res.get_forecast(steps=6)
fc_mean   = fc.predicted_mean
fc_ci_80  = fc.conf_int(alpha=0.20)
fc_ci_95  = fc.conf_int(alpha=0.05)
fc_mean.index   = range(LATEST_HIST, LATEST_HIST + 6)
fc_ci_80.index  = fc_mean.index
fc_ci_95.index  = fc_mean.index

# IEA STEPS 2030
steps_2030 = total_ev_stock_by(df, ["region_country","year","category"]) \
    .query("region_country == 'World' and category == 'Projection-STEPS' and year == 2030") \
    ["value"].sum()

fig_fc = go.Figure()

# 95% CI band
fig_fc.add_trace(go.Scatter(
    x=list(fc_ci_95.index) + list(fc_ci_95.index)[::-1],
    y=list(fc_ci_95.iloc[:,1]/1e6) + list(fc_ci_95.iloc[:,0]/1e6)[::-1],
    fill="toself", fillcolor="rgba(244,162,97,0.18)",
    line=dict(color="rgba(0,0,0,0)"),
    name="95% prediction interval", showlegend=True,
    hoverinfo="skip",
))
# 80% CI band
fig_fc.add_trace(go.Scatter(
    x=list(fc_ci_80.index) + list(fc_ci_80.index)[::-1],
    y=list(fc_ci_80.iloc[:,1]/1e6) + list(fc_ci_80.iloc[:,0]/1e6)[::-1],
    fill="toself", fillcolor="rgba(244,162,97,0.35)",
    line=dict(color="rgba(0,0,0,0)"),
    name="80% prediction interval", showlegend=True,
    hoverinfo="skip",
))
# historical
fig_fc.add_trace(go.Scatter(
    x=hist_stock.index, y=hist_stock.values/1e6,
    mode="lines+markers", name="Historical (IEA)",
    line=dict(color=PALETTE["primary"], width=3),
    marker=dict(size=7),
    hovertemplate="%{x}: %{y:.1f} M EVs<extra></extra>",
))
# ARIMA point forecast
fig_fc.add_trace(go.Scatter(
    x=fc_mean.index, y=fc_mean.values/1e6,
    mode="lines+markers", name="ARIMA forecast (mean)",
    line=dict(color=PALETTE["accent"], width=3, dash="dash"),
    marker=dict(size=8, symbol="diamond"),
    hovertemplate="%{x}: %{y:.1f} M EVs (forecast)<extra></extra>",
))
# IEA STEPS 2030 marker
fig_fc.add_trace(go.Scatter(
    x=[2030], y=[steps_2030/1e6],
    mode="markers+text", name="IEA STEPS 2030",
    marker=dict(size=18, color=PALETTE["danger"], symbol="star",
                line=dict(color="white", width=2)),
    text=[f"  IEA STEPS: {steps_2030/1e6:.0f} M"],
    textposition="middle right",
    hovertemplate="IEA STEPS 2030: %{y:.0f} M EVs<extra></extra>",
))

fig_fc.update_layout(
    title=dict(x=0.02, y=0.95, xanchor="left", yanchor="top", text="<b>World EV car stock - ARIMA forecast vs IEA STEPS</b>"),
    height=520, xaxis_title="Year",
    yaxis_title="EV cars on the road (millions)",
    hovermode="x unified",
    legend=dict(orientation="h", x=0.4, y=1, xanchor="left", yanchor="top"),
    margin=dict(l=20,r=200,t=80,b=70),
)
fig_fc.show()

print(f"\nARIMA 2030 forecast mean : {fc_mean.iloc[-1]/1e6:.1f} M EVs")
print(f"ARIMA 2030 80% interval  : {fc_ci_80.iloc[-1,0]/1e6:.1f} – {fc_ci_80.iloc[-1,1]/1e6:.1f} M")
print(f"IEA STEPS 2030           : {steps_2030/1e6:.1f} M EVs")


                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
ar.L1          1.0000      0.264      3.792      0.000       0.483       1.517
ma.L1          0.3195      0.286      1.115      0.265      -0.242       0.881
sigma2        7.4e+11   1.22e-13   6.07e+24      0.000     7.4e+11     7.4e+11



ARIMA 2030 forecast mean : 246.4 M EVs
ARIMA 2030 80% interval  : 208.3 – 284.6 M
IEA STEPS 2030           : 232.2 M EVs


**Tab 5 takeaways.**

- Our ARIMA and the IEA STEPS scenario are **in the same order of magnitude**, with ARIMA tending to be more aggressive when projected from 2024 alone (because it has no policy ceiling baked in). The IEA STEPS is policy-grounded; ARIMA is trend-grounded.

---

## Conclusions

1. **The hockey stick is real but the fleet lags the market.** EV sales-share is racing toward 20–25% globally while stock share is still ~5–6%. The 2020s are the *sales transition*; the 2030s will be the *fleet transition*.

2. **Two adoption archetypes coexist.** Norway-style markets win on intensity (saturated, BEV-dominant, high charger density). China-style markets win on volume (state-coordinated infrastructure + manufacturer scale). They are not the same playbook.

3. **Charging infrastructure is, globally, slightly behind EV deployment in 2024.** Several large markets have charger-to-EV ratios well above the comfort threshold — a real risk for new entrants and a real opportunity for charging-network operators.

4. **BEV is winning the long game**, but PHEV remains meaningful in markets with weak fast-charging networks or strong subsidies for them. As BEV TCO crosses ICE parity (likely 2026–28), PHEV share will compress.

5. **2030 STEPS is aggressive but not radical.** ARIMA-style trend extrapolation sits in the same neighborhood. The real risk to 2030 targets is **fleet turnover**, not sales penetration. Scrappage incentives and corporate fleet electrification are the high-leverage levers.

6. **Oil displacement compounds faster than the public realizes** — even at a conservative CAGR, the EV fleet displaces ~5–10 Mbd by 2030 in our extrapolation, which is structurally bearish for refined-product demand growth and bullish for grid investment.

---

### What we'd build next (analyst's note)

- **Cohort analysis by vintage** — track 2018 / 2020 / 2022 cohorts to estimate real fleet retirement curves rather than assume them.
- **Bottleneck modelling** — overlay battery-mineral supply forecasts (Li, Co, Ni, graphite) on the stock forecast.
- **Charger utilization** — chargers per EV is a stock metric; **kWh per charger per day** would tell us how stressed the network actually is.
- **Policy event-study** — formal causal inference around big policy events (IRA, China NEV mandate, EU 2035 ICE ban) instead of the eyeball method used here.
